## 自注意力机制Self-Attention

- RNN 或 LSTM 依靠“逐字递推”的机制，串行计算导致、长距离记忆容易遗忘、无法并行计算
- Transformer让模型能够同时“看到”整句话

自注意力机制的核心在于让输入序列中的每个词都能自主寻找与其他词的关联程度,引入三个矩阵来完成:
* Query ($Q$)：“我要寻找什么？”
* Key ($K$)：“我能提供什么相关信息？”
* Value ($V$)：“我实际包含的具体内容是什么？”

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

公式通俗解读：

   1. $Q \cdot K^T$：让每个词的 Query 去和所有词的 Key 做内积，算出彼此之间的“相关性得分”（注意力分数）。
   2. $\sqrt{d_k}$ 与 $\text{softmax}$：进行缩放和归一化，把得分变成概率分布（权重合为 1）。
   3. 乘以 $V$：用算出来的权重去对 Value 进行加权求和，从而让每个词都融合了整句话中与它相关的上下文知识。


### 用PyTorch实现 单头自注意力Self-Attention 矩阵运算

In [9]:
import torch
import torch.nn.functional as F

# 假设输入一句话包含 3 个词，每个词的向量维度是 4 (seq_len=3, d_model=4)
# 这一步模拟经典 NLP 中的词向量（Word Embeddings）输入
x = torch.randn(3, 4)
print("--- 输入 ---")
print(x)
# 1. 模拟三个线性变换层，初始化 W_q, W_k, W_v 权重矩阵
d_k = 4
W_q = torch.randn(4, d_k)
W_k = torch.randn(4, d_k)
W_v = torch.randn(4, d_k)

# 2. 映射得到 Q, K, V 矩阵
Q = torch.matmul(x, W_q)  # 形状: (3, 4)
K = torch.matmul(x, W_k)  # 形状: (3, 4)
V = torch.matmul(x, W_v)  # 形状: (3, 4)

# 3. 核心步骤 A：计算注意力分数 (Q * K^T) 并进行缩放
# K.t() 是 K 矩阵的转置
scores = torch.matmul(Q, K.t()) / torch.sqrt(torch.tensor(d_k, dtype=torch.float32))

# 4. 核心步骤 B：通过 Softmax 归一化，得到权重分布
attention_weights = F.softmax(scores, dim=-1)
print("--- 词与词之间的注意力权重矩阵 ---")
print(attention_weights)

# 5. 核心步骤 C：加权求和 Value，输出最终带有上下文信息的特征向量
output = torch.matmul(attention_weights, V)
print("\n--- 最终输出的特征特征向量 ---")
print(output)

--- 输入 ---
tensor([[-1.6348, -1.5878, -1.7563, -0.7798],
        [-1.7453, -1.6638, -1.5389,  0.2995],
        [ 1.0471,  0.1525, -1.7440, -0.9507]])
--- 词与词之间的注意力权重矩阵 ---
tensor([[1.2834e-03, 7.7759e-03, 9.9094e-01],
        [4.7889e-08, 1.9202e-07, 1.0000e+00],
        [1.8017e-01, 8.1983e-01, 3.3216e-09]])

--- 最终输出的特征特征向量 ---
tensor([[ 0.0800,  0.7975, -1.4994, -3.2743],
        [ 0.0571,  0.7910, -1.4868, -3.2941],
        [ 2.6121,  1.5083, -2.8569, -1.1879]])


上面的单头自注意力有一个致命缺陷：模型在同一时间只能聚焦于一种关联关系。

#### 多头注意力机制
为了让大模型具备更强大的特征提取能力，并能记住词语之间的先后顺序。则需要多头注意力机制（Multi-Head Attention）与位置编码（Positional Encoding）。

1. 为什么要“多头”?
    如果只有“一个头”，模型在看“苹果”这个词时，可能只能注意到它是一种“水果”（语义维度）。但如果引入“多个头”，不同的头就可以各自关注不同的维度 ：
    * Header 1：关注主谓宾等语法结构（如：“谁”吃了苹果）。
    * Header 2：关注词与词之间的语义关联（如：苹果属性是红色、脆的）。
    * Header 3：关注长距离指代（如：句子后半部分的“它”指的是苹果）。

2. 矩阵拆分与拼接的艺术
    “多头”并不是真的去训练几组完全独立的网络，而是将高维的空间切分成多个低维子空间。
    假设模型的总隐藏层维度 $d_{model} = 512$，我们设置有 $h = 8$ 个头。

    * 我们不会直接做 512 维的注意力计算。
    * 而是将 $Q, K, V$ 矩阵在特征维度上“切”成 8 份，每一份的维度是 $d_k = 512 / 8 = 64$ 。
    * 这 8 个大小为 64 维的头并行进行自注意力运算，算完之后，把 8 个头的输出在特征维度上重新拼接（Concat）起来，变回 512 维，最后乘上一个输出权重矩阵 $W^o$ 进行特征融合

3. 为什么自注意力机制“天生无序”？
    公式：$\text{Attention}(Q,K,V)$全是矩阵乘法和词与词之间的内积，如果你把输入句子的词语顺序完全打乱（例如把“我吃苹果”改成“苹果吃我”），只要词向量不变，模型算出来的注意力矩阵分布完全是一样的，只是位置换了。

    也就是说，自注意力机制本身是“词袋模型”，完全丧失了语序信息 。为了解决这个问题，必须在输入端强行注入位置信号 。

4. 正余弦位置编码
    Transformer 论文中采用了一种极其优雅的方法：使用不同频率的正弦（Sine）和余弦（Cosine）函数来计算绝对位置编码.
   **为什么要用三角函数？**
   因为三角函数具备相对位置对称性。通过高阶三角函数公式，$PE_{pos+k}$ 可以被 $PE_{pos}$ 线性表出。
   这意味着大模型不仅能知道某个词在第 3 个位置（绝对位置），还能轻易学会第 3 个位置和第 5 个位置之间相隔了 2 个单位（相对位置）。

   最重要的是：位置编码是直接与输入的词向量（Embedding）相加（Add）的，而不是拼接。

#### 用 PyTorch 编写多头注意力层

In [10]:
# 在工业界，大模型的代码里充满了各种张量形状变换。理解 view 和 transpose 是如何控制“多头”在矩阵中流转的。
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, "d_model 必须能被 num_heads 整除！"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # 每个头的维度

        # 定义 Q, K, V 的线性变换矩阵
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)

        # 最后的输出融合矩阵
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x):
        # x 形状: (batch_size, seq_len, d_model)
        batch_size, seq_len, d_model = x.size()

        # 1. 线性映射得到全局的 Q, K, V
        Q = self.W_q(x)  # (batch_size, seq_len, d_model)
        K = self.W_k(x)
        V = self.W_v(x)

        # 2. 核心魔法：将特征维度切分为多头，并转换维度以进行批量矩阵乘法
        # 变换顺序: (B, S, D) -> (B, S, H, D_K) -> (B, H, S, D_K)
        Q = Q.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)

        # 3. 计算每个头内部的 Scaled Dot-Product Attention
        # K.transpose(-2, -1) 将最后两维转置，形状变为 (batch_size, num_heads, d_k, seq_len)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_k ** 0.5)

        # 归一化得到注意力权重矩阵 (batch_size, num_heads, seq_len, seq_len)
        attn_weights = F.softmax(scores, dim=-1)

        # 与 V 相乘得到混合上下文特征 (batch_size, num_heads, seq_len, d_k)
        context = torch.matmul(attn_weights, V)

        # 4. 逆操作：把 8 个头的结果重新拼回高维空间
        # (B, H, S, D_K) -> (B, S, H, D_K) -> (B, S, D)
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        # 为什么要用 .transpose(1, 2)？如果不做转置直接用 view 会发生什么？（提示：直接 view 会把内存中连续的词语序列关系无脑切断，而转置能保证我们在各个“特征通道”上进行独立的矩阵相乘）。

        # 5. 通过最终线性层做特征映射
        output = self.W_o(context)

        return output, attn_weights

# --- 测试运行 ---
if __name__ == "__main__":
    # 模拟输入：1个 Batch，句子长 5 个词，每个词 128 维
    sample_input = torch.randn(1, 5, 128)

    # 实例化一个 4 头的注意力机制（每个头 128/4 = 32维）
    mha = MultiHeadAttention(d_model=128, num_heads=4)

    output, weights = mha(sample_input)
    print("输入形状:", sample_input.shape)
    print("输出形状 (应与输入一致):", output.shape)
    print("注意力权重矩阵形状 (Batch, Heads, Seq, Seq):", weights.shape)

输入形状: torch.Size([1, 5, 128])
输出形状 (应与输入一致): torch.Size([1, 5, 128])
注意力权重矩阵形状 (Batch, Heads, Seq, Seq): torch.Size([1, 4, 5, 5])


### 残差连接与层归一化
    多头注意力（MHA）只是在做“词与词的特征交互”，谁来做“词内部的特征深化”？
    在 Transformer 之前，深度神经网络一旦层数加深，就会遭遇梯度消失（Gradient Vanishing），导致模型根本无法训练。Transformer 能够堆叠几十甚至上百层。原理如下：

1. 残差连接（Residual Connection）
    公式极其简单：$\text{Output} = x + \text{SubLayer}(x)$
    每一层的输入 $x$，都会绕过这一层，直接加到这一层的输出上。在反向传播时，梯度的导数中会包含一个常数 $1$。这个“$1$”是一条绿色通道，保证了深层的梯度能够毫无损耗地传回输入层，彻底解决了深层网络死掉的问题。

2. 层归一化（LayerNorm）
    在计算机视觉中，Batch Normalization（批归一化）是绝对的主角。但在大语言模型（NLP）中，BN 是失效的：
   * **BatchNorm**：是在同一个 Batch（批次）内部，对所有样本的同一个特征维度进行归一化。然而，大模型的输入句子长度是动态变化的（有人一句话 5 个词，有人一句话 50 个词），在 Batch 维度上强行做均值和方差统计，会遭到长短句子的严重干扰。
   * **LayerNorm**：在单个样本（一句话）内部，对所有特征维度进行归一化。它完全独立于 Batch Size 和句子长度。无论一句话有多长，它都只在自己内部做归一化，非常稳定，天生适合处理序列文本。

   原始 Transformer 论文采用的是 Post-LN（先做残差再做归一化），但现代大模型（如 Llama、GPT-4）全部改为了 Pre-LN（先做归一化再做残差）。因为 Pre-LN 允许梯度直接穿透主干网络，训练稳定性呈指数级上升。


#### 位置前馈神经网络Position-wise FFN
很多人会忽略 FFN，觉得它只是普通的线性层。但实际上，它是大模型的“知识存储库”。

* 多头注意力层（MHA）的作用：是让词与词之间发生关系（空间交互）。比如让“苹果”注意到后文的“好吃”。但它主要做的是线性变换。
* 前馈神经网络层（FFN）的作用：对每一个位置的词向量独立进行非线性变换。它不关心别的词，只专注于深化当前词自身的特征。

标准 Transformer 中的 FFN 由两个线性层和一个激活函数（如 ReLU 或 GELU）组成：
$$\text{FFN}(x) = \max(0, xW_1 + b_1)W_2 + b_2$$
架构设计：
这里的首个线性层 $W_1$ 会把向量维度放大 4 倍（例如 512 维放大到 2048 维），通过激活函数做非线性映射后，再由 $W_2$ 缩小回原维度（2048 维缩回 512 维）。这种“先升维再降维”的结构，被普遍认为是大模型存储隐式“世界知识”的地方。

我们将MHA多头注意力逻辑与 LayerNorm归一化、前馈神经网络FFN 融合在一起（这里直接调用 PyTorch 官方的 `nn.MultiheadAttention` 保持代码简洁易读），采用现代大模型普遍使用的 Pre-LN 架构进行组装。

In [ ]:
# 代码断点调试：运行上面的 PyTorch 代码，打印出 attn_output 和 ffn_output 的形状，验证模型在穿过整个 Block 的过程中，张量的形状是否全程保持为 (2, 10, 512)。这种“形状不变性”如何帮助我们无限制地堆叠层数？
import torch
import torch.nn as nn

class TransformerEncoderLayerPreLN(nn.Module):
    def __init__(self, d_model, num_heads, dim_feedforward, dropout=0.1):
        super(TransformerEncoderLayerPreLN, self).__init__()

        # 1. 多头注意力子层
        self.self_attn = nn.MultiheadAttention(embed_dim=d_model, num_heads=num_heads, batch_first=True)

        # 2. 前馈神经网络子层 (升维4倍再降维)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.GELU(), # 现代模型更常用 GELU 替代 ReLU
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, d_model)
        )

        # 3. 两个层归一化组件 (Pre-LN 架构)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x 的形状: (batch_size, seq_len, d_model)

        # --- 第一阶段：MHA 模块 (带 Pre-LN 与残差连接) ---
        # 1. 先进行层归一化 (Pre-LN)
        norm_x1 = self.norm1(x)
        # 2. 计算自注意力 (MHA 期待的输入是 Q, K, V，在自注意力中三者皆为 norm_x1)
        attn_output, _ = self.self_attn(norm_x1, norm_x1, norm_x1)
        # 3. 残差连接：原始的 x + 变换后的输出
        x = x + self.dropout(attn_output)

        # --- 第二阶段：FFN 模块 (带 Pre-LN 与残差连接) ---
        # 4. 再次进行层归一化 (Pre-LN)
        norm_x2 = self.norm2(x)
        # 5. 通过前馈神经网络提取非线性特征
        ffn_output = self.ffn(norm_x2)
        # 6. 残差连接
        x = x + self.dropout(ffn_output)

        return x

# --- 测试运行 ---
if __name__ == "__main__":
    # 参数定义：维度512，8个头，FFN隐层维度2048 (4倍)
    d_model = 512
    num_heads = 8
    dim_ff = 2048

    # 实例化我们的 Transformer 编码器块
    encoder_layer = TransformerEncoderLayerPreLN(d_model, num_heads, dim_ff)

    # 模拟输入：Batch大小为2，句子长度为10，词向量维度512
    fake_input = torch.randn(2, 10, 512)

    # 前向传播
    output = encoder_layer(fake_input)

    print("--- 组装测试成功 ---")
    print("输入张量形状:", fake_input.shape)
    print("输出张量形状 (完美保持一致):", output.shape)

    # 架构逆向思考：尝试在代码中将 Pre-LN 改回原始的 Post-LN（即：先做 attn_output，然后 x = self.norm1(x + attn_output)）。思考并查阅一下资料，为什么在百亿或千亿参数量下，Post-LN 会让大模型在训练初期直接产生“梯度爆炸”或不可控的“Nan”？

#### 掌握生成式模型的灵魂——因果掩码与自回归机制
从“理解文本”的 Encoder 跨越到“生成文本”的 Decoder-only 架构。如果你想搞懂 GPT-4、Llama 3、Qwen 是如何做到像人类一样逐字吐出文本（Autoregressive 序列生成）的，因果掩码（Causal Masking）就是其中的灵感核心。
两个核心逻辑：

1. 为什么 GPT 这种大模型在训练和推理时，绝对不能让前面的词看到后面的词？
2. 如何在矩阵并行计算的同时，通过“作弊”手段优雅地蒙住模型的眼睛（Causal Mask）？

###### 双向看（Encoder）与单向看（Decoder）
Encoder 块，其自注意力是双向（Bi-directional）的。当模型处理“我喜欢吃苹果”时，“我”在计算注意力时可以同时看到“喜欢”、“吃”、“苹果”。这非常适合做文本分类、情感分析或实体抽取（比如早期的 BERT 模型）。

但是，对于像 GPT 这样的生成式大模型，它的核心任务是自回归生成（Autoregressive Generation）——根据历史文本预测下一个词：
* 输入：“我” $\rightarrow$ 预测下一个词：“喜欢”
* 输入：“我 喜欢” $\rightarrow$ 预测下一个词：“吃”
* 输入：“我 喜欢 吃” $\rightarrow$ 预测下一个词：“苹果”

这就带来了一个致命的训练矛盾：
为了让 GPU 能够并行训练，我们是一口气把整句“我喜欢吃苹果”作为输入喂给模型的。如果还用昨天的双向自注意力，当模型在预测“喜欢”时，它其实已经通过矩阵运算偷偷看到了后面的“吃”和“苹果”。这就是严重的“标签泄露（Data Leakage）”，会导致模型在训练时全对，到了真正推理生成时直接抓瞎。

为了解决这个问题，我们必须在训练时引入因果掩码（Causal Masking），强行遮住未来的词。

###### 因果掩码的数学魔法
如何既能保持整句话矩阵并行计算的高效，又能不让前面的词看到后面的词呢？Transformer 论文引入了一个极其巧妙的下三角矩阵。假设我们的句子包含 4 个词，通过 $Q \cdot K^T / \sqrt{d_k}$ 算出来的原始注意力得分（Attention Scores）是一个 $4 \times 4$ 的矩阵：

* 第 $i$ 行代表当前正在处理的词，第 $j$ 列代表被关注的词。
* 只要 $j > i$（即右上三角区域），就意味着第 $j$ 个词相对于第 $i$ 个词是未来的词。
* 我们在将这个矩阵送进 Softmax 归一化之前，强行把右上三角区域的所有分数值替换为 $-\infty$（负无穷，工程上通常用一个极小的负数如 -1e9 表示）。
$$\text{Scores}_{\text{masked}} = \begin{pmatrix} 12.5 & -\infty & -\infty & -\infty \\ 8.2 & 14.1 & -\infty & -\infty \\ 3.1 & 9.4 & 11.2 & -\infty \\ 4.5 & 7.2 & 13.1 & 15.6 \end{pmatrix}$$

当我们对这一行行数据做 $\text{softmax}$ 激活时，因为 $e^{-\infty} = 0$，那些未来词的注意力权重会精确地变成 0！
这样，第一行（第一个词）的注意力就只能加权在自己身上；第四行（第四个词）则可以融合前四个词的所有上下文。



#### 徒手实现带 Mask 的多头注意力层
在上面代码 `nn.MultiheadAttention` 基础上进行扩展，利用 PyTorch 的 torch.tril（提取下三角矩阵）以及 masked_fill 动态注入因果掩码。代码如下：

In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CausalMultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(CausalMultiHeadAttention, self).__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x):
        batch_size, seq_len, d_model = x.size()

        # 1. 映射并切分为多头 (B, H, S, D_K)
        Q = self.W_q(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)

        # 2. 计算原始注意力得分，形状: (batch_size, num_heads, seq_len, seq_len)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_k ** 0.5)

        # 3. 核心魔法：生成因果掩码 (Causal Mask)
        # torch.ones(seq_len, seq_len) 创建全 1 矩阵
        # torch.tril 取出其下三角部分，上三角变成 0
        # 形状为 (seq_len, seq_len) 的下三角矩阵，1 表示保留，0 表示遮掩
        mask = torch.tril(torch.ones(seq_len, seq_len, device=x.device))

        # 4. 将未来词位置 (mask == 0 的地方) 强行填入负无穷 (-1e9)
        # 这里借助了 PyTorch 的广播机制，mask 会自动适配 Batch 和 Head 维度
        scores = scores.masked_fill(mask == 0, -1e9)

        # 5. 通过 Softmax 归一化，此时未来词的权重会变成 0
        attn_weights = F.softmax(scores, dim=-1)

        # 6. 加权求和并拼回原高维特征空间
        context = torch.matmul(attn_weights, V)
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)

        output = self.W_o(context)
        return output, attn_weights

# --- 测试运行 ---
if __name__ == "__main__":
    # 模拟输入：1个样本，句子包含 4 个词，每个词 16 维
    fake_tokens = torch.randn(1, 4, 16)

    causal_mha = CausalMultiHeadAttention(d_model=16, num_heads=2)
    output, weights = causal_mha(fake_tokens)

    print("--- 因果掩码矩阵验证 ---")
    # 打印第一个 Batch 里的第一个 Head 的注意力权重矩阵
    print(weights[0, 0].detach().numpy().round(4))


--- 因果掩码矩阵验证 ---
[[1.     0.     0.     0.    ]
 [0.2492 0.7508 0.     0.    ]
 [0.2275 0.2852 0.4873 0.    ]
 [0.1778 0.2658 0.2145 0.3419]]


1. 肉眼验证下三角矩阵：运行上面的代码，仔细观察终端打印出来的权重矩阵（$4 \times 4$）。是不是右上角（对角线以上）的值全部都是精确的 0.0000？如果第 0 行（第一个词）只有第一列有值（为 1），这在物理意义上代表了什么？
2. 为未来埋下伏笔（KV Cache 的思考）：大模型在真正的自回归推理（Generation）时，是一步一步往后预测的。比如当序列长度从 4 变成 5 时，前 4 个词的 $K$ 和 $V$ 矩阵其实已经算过了且由于因果掩码的存在，它们并不会受到第 5 个词的干扰。那么在工程上，我们每次预测新词时，有没有必要重新计算前 4 个词的 $K$ 和 $V$ 呢？如果不重新算，应该怎么把它存起来？（提示：这就是目前所有大模型推理加速的核心技术——KV Cache（键值缓存）。

## 拼装完整 Decoder 模型，实现自回归生成循环
1. 宏观合围：理解一个完整的 Decoder-only 大模型，从输入的 Token ID 到最后输出 Vocabulary 概率分布的端到端数据流（Pipeline）。
2. 让模型开口说话：在工程上徒手写出大模型的 generate() 函数，搞懂推理时“逐字吐词”的循环逻辑。

###### Decoder-only 大模型的端到端全景图
现代大模型（如 GPT-4、Llama 3、Mistral、Qwen）几乎清一色采用了 Decoder-only 架构。它的端到端向前传播（Forward）流程可以用以下这张全景图来概括：
**数据的奇幻漂流：从文本到 Logits**
1. Tokenization（分词）：输入的文本（如“我喜欢”）被切分成 Token，并转换成一串整数 ID（如 [102, 453, 899]）。
2. Embedding 层：
    * Token Embedding：将每个 ID 映射成一个高维向量（$d_{model}$）。
    * Positional Embedding：生成对应的位置向量，与 Token 向量直接相加，注入顺序信息。
3. Transformer Blocks 堆叠：带位置信息的向量进入主干网络。网络中堆叠了 $N$ 层我们实现的 Causal Transformer Layers（带因果掩码的层）。文本特征在这里经过层层深化，每个词都融合了它之前所有历史词的上下文。
4. Final LayerNorm：在所有 Block 结束后，再做一次全局的层归一化。
5. LM Head（语言模型头）：这是一个最普通的、不带偏置的线性层（nn.Linear(d_model, vocab_size)）。它将最终的隐藏特征维度（如 4096）映射回你的词表大小（Vocab Size，如 32000）。
    * Logits：LM Head 输出的原始未归一化得分。Logits 的形状为 (batch_size, seq_len, vocab_size)，它代表了模型认为在当前上下文下，词表里每一个词作为“下一个词”出现的可能性。

###### 自回归生成循环（Autoregressive Loop）
在训练时，我们通过因果掩码，利用矩阵乘法一次性并行计算了整句话的损失（Loss）。但在实际推理（Inference/Generation）时，大模型必须像人类写字一样，一个词一个词地往后吐。
自回归生成的标准算法逻辑：
1. 初始输入输入序列：["我", "喜欢"]
2. 将序列喂给模型 $\rightarrow$ 模型输出一堆 Logits $\rightarrow$ 我们只取出最后一个位置的 Logits（代表对下一个词的预测）。
3. 经过策略选择（如贪婪搜索选概率最大的词），选出了 "吃"。
4. 将新词拼接到原序列后面，序列变成 ["我", "喜欢", "吃"]。
5. 再次把新序列喂给模型 $\rightarrow$ 预测出 "苹果"。
6. 重复此过程，直到模型吐出特殊的结束符 <|endoftext|> (EOS Token) 或达到了我们设定的最大长度。



#### 从零拼装 Mini-GPT 并实现 generate 函数
为了让你有最直观的体验，我们不用任何封装，直接用最底层的 PyTorch 算子，把词表映射、多层堆叠以及自回归循环全部写在一起。

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# =====================================================================
# 1. 组装昨天的因果注意力与前馈网络，形成单个 Decoder Block
# =====================================================================
class CausalTransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, dim_feedforward):
        super().__init__()
        # 为了保证本核心代码整洁，此处直接调用 PyTorch 官方 MHA
        self.attn = nn.MultiheadAttention(embed_dim=d_model, num_heads=num_heads, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.GELU(),
            nn.Linear(dim_feedforward, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        seq_len = x.size(1)
        # 生成因果掩码：PyTorch 的 MultiheadAttention 接收的 attn_mask 如果是 bool 型
        # True 代表要遮掩（Mask），False 代表不遮掩。这与我们昨天的 masked_fill(mask==0, -1e9) 物理意义一致
        mask = torch.triu(torch.ones(seq_len, seq_len, device=x.device), diagonal=1).bool()

        # Pre-LN 架构
        norm_x = self.norm1(x)
        # attn_mask 传入因果掩码，阻止未来信息泄露
        attn_out, _ = self.attn(norm_x, norm_x, norm_x, attn_mask=mask, need_weights=False)
        x = x + attn_out

        x = x + self.ffn(self.norm2(x))
        return x

# =====================================================================
# 2. 串联全局，构建完整的 Mini-GPT 模型
# =====================================================================
class MiniGPT(nn.Module):
    def __init__(self, vocab_size, max_seq_len, d_model, num_heads, dim_ff, num_layers):
        super().__init__()
        self.max_seq_len = max_seq_len

        # Token 与位置嵌入层（此处采用 GPT-2 式的可学习位置编码）
        self.token_embeddings = nn.Embedding(vocab_size, d_model)
        self.position_embeddings = nn.Embedding(max_seq_len, d_model)

        # 堆叠多个 Causal Transformer Block
        self.blocks = nn.ModuleList([
            CausalTransformerBlock(d_model, num_heads, dim_ff) for _ in range(num_layers)
        ])

        self.final_norm = nn.LayerNorm(d_model)
        # LM Head：将隐藏层维度映射回词表大小
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, idx):
        # idx 形状: (batch_size, seq_len)
        b, t = idx.size()
        assert t <= self.max_seq_len, "输入序列长度超过了模型的最大承受范围！"

        # 生成位置 ID：[[0, 1, 2, ... t-1]]
        pos = torch.arange(0, t, dtype=torch.long, device=idx.device).unsqueeze(0)

        # 向量相加：Token Embedding + Position Embedding
        x = self.token_embeddings(idx) + self.position_embeddings(pos)

        # 穿过每一层 Block
        for block in self.blocks:
            x = block(x)

        x = self.final_norm(x)
        logits = self.lm_head(x) # 形状: (b, t, vocab_size)
        return logits

    # =====================================================================
    # 3. 核心精讲：实现自回归生成循环 (Greedy Search)
    # =====================================================================
    @torch.no_grad() # 推理时不需要计算梯度
    def generate(self, idx, max_new_tokens):
        # idx 是初始给大模型的 Prompt Token ID，形状: (batch_size, seq_len)
        for _ in range(max_new_tokens):
            # 1. 裁剪输入：如果当前长度超过最大限制，截断左侧（防止位置编码越界）
            idx_cond = idx[:, -self.max_seq_len:]

            # 2. 前向传播拿到当前序列所有词的 Logits
            logits = self.forward(idx_cond)

            # 3. 核心步骤：我们只聚焦于“最后一个位置” (t=-1) 的预测结果
            # logits[:, -1, :] 形状变为 (batch_size, vocab_size)
            next_token_logits = logits[:, -1, :]

            # 4. 贪婪策略（Greedy Search）：直接取概率最大的那个 Token ID
            next_token = torch.argmax(next_token_logits, dim=-1, keepdim=True)

            # 5. 自回归拼接：将新诞生的 token 追加到输入序列的末尾，成为下一次循环的“历史”
            idx = torch.cat((idx, next_token), dim=1)

        return idx

# --- 测试运行 ---
if __name__ == "__main__":
    # 初始化一个超微型 Mini-GPT
    # 词表大小 1000，最大支持长度 32，维度 64，2个头，堆叠 3 层 Block
    model = MiniGPT(vocab_size=1000, max_seq_len=32, d_model=64, num_heads=2, dim_ff=256, num_layers=3)
    model.eval() # 切换到评估推理模式

    # 模拟初始 Prompt：假设分词器把“人工智能”切成了 3 个 Token ID
    prompt_tokens = torch.tensor([[12, 45, 88]], dtype=torch.long)
    print("初始 Prompt Token 序列:", prompt_tokens.tolist()[0])

    # 让模型续写 5 个新词
    generated_sequence = model.generate(prompt_tokens, max_new_tokens=5)

    print("生成后的完整 Token 序列:", generated_sequence.tolist()[0])
    print("大模型独立吐出的 5 个新词:", generated_sequence.tolist()[0][-5:])

初始 Prompt Token 序列: [12, 45, 88]
生成后的完整 Token 序列: [12, 45, 88, 15, 517, 374, 816, 308]
大模型独立吐出的 5 个新新词: [15, 517, 374, 816, 308]


**思考**：
看到终端里原序列通过 torch.cat 不断变长、吐出新 Token 的那一刻，你已经完成了工业界大模型最底层的核心解构！
请尝试完成以下两个极具深度的思考题，这能为你下周正式进入“大模型的训练与微调（预训练、SFT、LoRA）”打下极其恐怖的坚实基础：
1. 直观感受生成效率（KV Cache 的思想起源）：
在上面的 generate 代码中，当循环进行到第 5 次时，序列长度变成了 7。大模型在做 self.forward(idx_cond) 时，会把这 7 个词全部重新做一遍矩阵乘法、LayerNorm 和 FFN。
    * 思考：前 6 个词由于因果掩码的存在，它们算出来的 $K$ 和 $V$ 矩阵其实和上一次循环完全一模一样。我们这种每次都“重头全算一遍”的贪婪生成逻辑，在处理超长文本（如 4K 窗口）时，会遇到什么性能瓶颈？

2. 生成多样性的秘密（从 Greedy Search 到 Sampling）：
    在代码第 4 步里，我们使用了绝对理性的 torch.argmax（贪婪搜索）。这会导致只要 Prompt 固定，模型每次吐出的字就完全一模一样。如果我们在把 Logits 变成概率之前，给它除以一个常数 $T$（Temperature，温度），然后再用 torch.multinomial 按照概率去随机采样，为什么就能让大模型具备“创造力”和“每次回答都不一样”的特性？如果温度 $T \to 0$ 或者 $T \to \infty$，模型的表现会分别发生什么好玩的改变？